# 🚗 Used Car Price Prediction: End-to-End Machine Learning Pipeline

**Course:** Machine Learning Mini Project | BE Computer Engineering  
**Problem Type:** Supervised Learning (Regression)  
**Goal:** Predict used vehicle resale prices (`selling_price` in INR) based on vehicle characteristics, compare classical ML and ensemble regression models, perform exploratory data analysis, and build an inference pipeline.

---
### 📋 Table of Contents (8 Core Components)
1. [Component 1: Problem Definition & Objectives](#c1)
2. [Component 2: Dataset Loading & Overview](#c2)
3. [Component 3: Data Preprocessing & Feature Engineering](#c3)
4. [Component 4: Exploratory Data Analysis (EDA)](#c4)
5. [Component 5: Model Pipeline & Architecture](#c5)
6. [Component 6: Model Training & Evaluation](#c6)
7. [Component 7: Model Comparison & Visual Diagnostics](#c7)
8. [Component 8: Inference Engine & Deployment Demo](#c8)

---
## ⚙️ Environment Setup & Google Colab Helpers
Run the cell below to import all necessary libraries and configure Matplotlib/Seaborn aesthetics.

In [ ]:
# Import core packages
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn tools
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import joblib

# Visual formatting
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('deep')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('✅ Packages successfully imported!')

<a id='c1'></a>
## 📌 Component 1: Problem Definition & Objectives

Estimating the fair resale price of a pre-owned vehicle is difficult because prices vary across multiple interacting attributes:
- **Vehicle Age**: Natural depreciation over time.
- **Mileage / Usage Intensity**: Wear-and-tear through total kilometres driven and annual usage rate.
- **Brand Equity & Prestige**: Luxury marques vs economy commuter brands.
- **Drivetrain & Fuel Type**: Diesel, Petrol, CNG, Electric, Transmission type.
- **Ownership History**: 1st, 2nd, 3rd, or 4th+ owners.

**Project Objectives:**
1. Build an end-to-end reproducible machine learning regression pipeline.
2. Train, optimize, and benchmark 5 distinct regression models under 5-Fold Cross Validation.
3. Log-transform the right-skewed target variable to stabilize variance and prevent prediction bias.
4. Deliver an inference function capable of predicting fair market prices for new car listings.

<a id='c2'></a>
## 📊 Component 2: Dataset Loading & Inspection

We use the **CarDekho Used Car Dataset** containing historical listings of vehicles sold in India.
If running in Google Colab, you can upload `car_data.csv` using the file upload button on the left sidebar, or let the code load it locally.

In [ ]:
# Load dataset (handles local workspace, Colab current directory, or parent data/ folder)
DATA_CANDIDATES = [
    Path('data/car_data.csv'),
    Path('car_data.csv'),
    Path('../data/car_data.csv')
]

data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)

if data_path is None:
    print('⚠️ car_data.csv not found locally. If on Colab, please upload car_data.csv to the files tab!')
    from google.colab import files
    uploaded = files.upload()
    data_path = Path(list(uploaded.keys())[0])

df_raw = pd.read_csv(data_path)
print(f'✅ Dataset loaded successfully!')
print(f'Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
df_raw.head(10)

In [ ]:
# Dataset Structure & Missing Values Check
print('=== Data Types & Non-Null Counts ===')
print(df_raw.info())
print('\n=== Summary Statistics (Numerical) ===')
display(df_raw.describe().T)
print('\n=== Missing Values Count ===')
print(df_raw.isnull().sum())

<a id='c3'></a>
## 🛠️ Component 3: Data Preprocessing & Feature Engineering

To prepare the raw data for modeling, we perform the following transformations:
1. **Deduplication**: Drop exact duplicate rows.
2. **Brand Extraction**: Extract the manufacturer name (e.g. `'Maruti'`, `'Hyundai'`, `'BMW'`) from the full vehicle `name` string.
3. **Car Age**: Convert manufacturing `year` into `car_age = CURRENT_YEAR - year`.
4. **Usage Intensity (`km_per_year`)**: Calculate annual kilometres driven (`km_driven / car_age`) to distinguish highway runners from city cars.
5. **Outlier Filtering**: Drop physically invalid rows and trim extreme top 1% outliers in price and mileage.

In [ ]:
CURRENT_YEAR = 2026
TARGET = 'selling_price'

def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.drop_duplicates().dropna().copy()
    
    # 1. Extract brand
    if 'name' in df_clean.columns:
        df_clean['brand'] = df_clean['name'].astype(str).str.split().str[0].str.title()
        df_clean = df_clean.drop(columns=['name'])
    
    # 2. Derive car age
    if 'year' in df_clean.columns:
        df_clean['car_age'] = CURRENT_YEAR - df_clean['year']
        df_clean = df_clean.drop(columns=['year'])
        
    # 3. Usage intensity (km per year)
    df_clean['km_per_year'] = (df_clean['km_driven'] / df_clean['car_age'].clip(lower=1)).round(2)
    
    # 4. Outlier trimming (top 1% tails)
    df_clean = df_clean[(df_clean[TARGET] > 0) & (df_clean['km_driven'] >= 0) & (df_clean['car_age'] >= 0)]
    for col in [TARGET, 'km_driven']:
        df_clean = df_clean[df_clean[col] <= df_clean[col].quantile(0.99)]
        
    return df_clean.reset_index(drop=True)

df = preprocess_data(df_raw)
print(f'Raw Shape: {df_raw.shape}  -->  Cleaned Shape: {df.shape}')
df.head()

<a id='c4'></a>
## 📈 Component 4: Exploratory Data Analysis (EDA)

Let's visualize the key distributions, pricing relationships, and categorical correlations.

In [ ]:
# 1. Target Distribution (Raw vs Log Transform)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df[TARGET], kde=True, ax=axes[0], color='#3b82f6', bins=30)
axes[0].set_title('Raw Selling Price Distribution (Right-Skewed)')
axes[0].set_xlabel('Price in INR (₹)')

sns.histplot(np.log1p(df[TARGET]), kde=True, ax=axes[1], color='#10b981', bins=30)
axes[1].set_title('Log-Transformed Price (Normal Bell Curve)')
axes[1].set_xlabel('log1p(Selling Price)')

plt.tight_layout()
plt.show()

In [ ]:
# 2. Price vs Age and Kilometres Driven
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x='car_age', y=TARGET, hue='fuel', alpha=0.6, ax=axes[0])
axes[0].set_title('Resale Price vs Vehicle Age')
axes[0].set_xlabel('Car Age (Years)')
axes[0].set_ylabel('Selling Price (INR)')

sns.scatterplot(data=df, x='km_driven', y=TARGET, hue='transmission', alpha=0.6, ax=axes[1])
axes[1].set_title('Resale Price vs Odometer (km driven)')
axes[1].set_xlabel('Kilometres Driven')
axes[1].set_ylabel('Selling Price (INR)')

plt.tight_layout()
plt.show()

In [ ]:
# 3. Median Price by Top Brands & Correlation Heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_brands = df.groupby('brand')[TARGET].median().sort_values(ascending=False).head(12)
sns.barplot(x=top_brands.values, y=top_brands.index, ax=axes[0], palette='crest')
axes[0].set_title('Top 12 Brands by Median Resale Price')
axes[0].set_xlabel('Median Selling Price (INR)')

num_cols = ['selling_price', 'car_age', 'km_driven', 'km_per_year']
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', ax=axes[1])
axes[1].set_title('Numerical Correlation Heatmap')

plt.tight_layout()
plt.show()

<a id='c5'></a>
## 🏗️ Component 5: Model Pipeline & Architecture

To prevent data leakage, all preprocessing is encapsulated inside a `ColumnTransformer` and Scikit-Learn `Pipeline` fitted **strictly on the training split**.

### Pipeline Structure:
1. **Numeric Pipeline**: `StandardScaler()` applied to `['km_driven', 'car_age', 'km_per_year']`.
2. **Categorical Pipeline**: `OneHotEncoder(handle_unknown='ignore')` applied to `['brand', 'fuel', 'seller_type', 'transmission', 'owner']`.
3. **Target Transformation**: Wrapped in `TransformedTargetRegressor(func=np.log1p, inverse_func=np.expm1)` to train on log-price and predict back into original Rupees.

In [ ]:
NUMERIC_COLS = ['km_driven', 'car_age', 'km_per_year']
CATEGORICAL_COLS = ['brand', 'fuel', 'seller_type', 'transmission', 'owner']
SEED = 42

def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer([
        ('num', StandardScaler(), NUMERIC_COLS),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_COLS),
    ])

def get_models() -> dict:
    regressors = {
        'Linear Regression': LinearRegression(),
        'Ridge Regression': Ridge(alpha=1.0, random_state=SEED),
        'Decision Tree': DecisionTreeRegressor(max_depth=8, min_samples_split=10, min_samples_leaf=4, random_state=SEED),
        'Random Forest (Tuned)': RandomForestRegressor(n_estimators=300, max_depth=14, min_samples_split=6, min_samples_leaf=2, max_features='sqrt', n_jobs=-1, random_state=SEED),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.06, max_depth=5, min_samples_split=6, min_samples_leaf=3, subsample=0.85, random_state=SEED),
    }
    
    pipelines = {}
    for name, model in regressors.items():
        pipe = Pipeline([('prep', build_preprocessor()), ('model', model)])
        pipelines[name] = TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)
    return pipelines

print('✅ Model Pipelines constructed successfully!')

<a id='c6'></a>
## 🧪 Component 6: Model Training & Evaluation

We split the data **80% Train / 20% Test** with `random_state=42` and evaluate every model using:
- **$R^2$ Score**: Proportion of variance explained.
- **5-Fold Cross-Validation $R^2$**: Validating generalizability across folds.
- **MAE (Mean Absolute Error)**: Average prediction error in INR.
- **RMSE (Root Mean Squared Error)**: Standard deviation of residuals.

In [ ]:
X = df[NUMERIC_COLS + CATEGORICAL_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f'Training Set: {X_train.shape[0]} samples | Test Set: {X_test.shape[0]} samples')

results_list = []
fitted_models = {}

for name, model in get_models().items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    fitted_models[name] = model
    
    # Predictions
    y_pred = model.predict(X_test)
    y_train_pred = model.predict(X_train)
    
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    results_list.append({
        'Model': name,
        'Test R2': round(r2, 4),
        '5-Fold CV R2': round(cv_scores.mean(), 4),
        'Train R2': round(train_r2, 4),
        'MAE (₹)': round(mae, 2),
        'RMSE (₹)': round(rmse, 2),
    })

results_df = pd.DataFrame(results_list).sort_values('Test R2', ascending=False).reset_index(drop=True)
print('\n=== Final Model Benchmarking Table ===')
display(results_df)

<a id='c7'></a>
## 🔬 Component 7: Model Comparison & Visual Diagnostics

Let's visualize the **Actual vs. Predicted** scatter plot and calculate **Permutation Feature Importance**.

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = fitted_models[best_name]
y_pred_best = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Actual vs Predicted
axes[0].scatter(y_test / 100000, y_pred_best / 100000, alpha=0.45, color='#3b82f6', edgecolors='none')
max_val = max(y_test.max(), y_pred_best.max()) / 100000
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction Line')
axes[0].set_title(f'Actual vs Predicted Price ({best_name})')
axes[0].set_xlabel('Actual Price (Lakhs ₹)')
axes[0].set_ylabel('Predicted Price (Lakhs ₹)')
axes[0].legend()

# 2. Permutation Feature Importance
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=SEED, n_jobs=-1)
imp_series = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=True)
imp_series.plot(kind='barh', color='#10b981', ax=axes[1])
axes[1].set_title(f'Feature Importance ({best_name})')
axes[1].set_xlabel('Mean Decrease in R² Score')

plt.tight_layout()
plt.show()

<a id='c8'></a>
## 🚀 Component 8: Inference Engine & Deployment Demo

Here we create an end-to-end prediction function that takes raw vehicle attributes, calculates derived features (`car_age`, `km_per_year`), predicts the fair market resale price, and provides a market valuation interval ($\pm 7.5\%$).

In [ ]:
def predict_car_price(brand: str, year: int, km_driven: int, fuel: str, transmission: str, owner: str, seller_type: str = 'Individual'):
    car_age = CURRENT_YEAR - year
    km_per_year = round(km_driven / max(car_age, 1), 2)
    
    input_df = pd.DataFrame([{
        'brand': brand,
        'fuel': fuel,
        'seller_type': seller_type,
        'transmission': transmission,
        'owner': owner,
        'car_age': car_age,
        'km_driven': km_driven,
        'km_per_year': km_per_year
    }])
    
    pred_price = float(best_model.predict(input_df)[0])
    pred_price = max(pred_price, 25000)
    
    # Price band +/- 7.5%
    margin = pred_price * 0.075
    low_band = pred_price - margin
    high_band = pred_price + margin
    
    print('==================================================')
    print(f' 🚗 Valuation for {year} {brand} ({fuel}, {transmission})')
    print('==================================================')
    print(f' • Estimated Fair Resale Price: ₹{pred_price:,.0f} ({pred_price/100000:.2f} Lakh)')
    print(f' • Fair Market Band:            ₹{low_band:,.0f} – ₹{high_band:,.0f}')
    print(f' • Usage Intensity:             {km_per_year:,.0f} km/year')
    print(f' • Active Model:                {best_name}')
    print('==================================================\n')

# Test Demo Cases
predict_car_price(brand='Maruti', year=2021, km_driven=35000, fuel='Petrol', transmission='Manual', owner='First Owner')
predict_car_price(brand='Hyundai', year=2019, km_driven=55000, fuel='Diesel', transmission='Automatic', owner='First Owner')
predict_car_price(brand='Audi', year=2018, km_driven=40000, fuel='Diesel', transmission='Automatic', owner='Second Owner')

---
## 💾 Save Artifacts
Exporting the best model and metadata for web deployment.

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/best_model.pkl')

meta = {
    'best_model': best_name,
    'numeric': NUMERIC_COLS,
    'categorical': CATEGORICAL_COLS,
    'choices': {c: sorted(df[c].astype(str).unique().tolist()) for c in CATEGORICAL_COLS},
    'ranges': {
        'km_driven': [0, int(df['km_driven'].max())],
        'car_age': [0, int(df['car_age'].max())],
        'km_per_year': [0, int(df['km_per_year'].max())]
    }
}
with open('models/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ Artifacts successfully saved to models/best_model.pkl and models/meta.json!')